In [2]:
import pandas as pd
import numpy as np

In [3]:
import talib  # библиотека для расчёта индикаторов

In [4]:
from utils import parabolic_sar

In [5]:
N = 20

In [6]:
# 1. Загрузка данных
# Предполагаем, что есть CSV с колонками: datetime, open, high, low, close, volume
# df = pd.read_csv('dowload_data/CNYRUBF_1Min.txt', parse_dates=['datetime'], index_col='datetime')
def readQuotes(path):
    try:
        quotes = pd.read_csv(path, sep=';')
        quotes['date'] = quotes['date'] + ' ' + quotes['time']
        del quotes['time']
        quotes['date'] = pd.to_datetime(quotes['date'], format='%d.%m.%Y %H:%M:%S', errors='raise', dayfirst=True)
        quotes.index = quotes['date']
        quotes.index.name = 'Date'
        del quotes['date']
        quotes[quotes.columns].astype(float)
        return quotes
    except pd.errors.EmptyDataError:
        print(f"File {path} is empty")
        return pd.DataFrame()
    except pd.errors.ParserError as e:
        print(f"Error parsing file {path}: {e}")

In [7]:
df_history = readQuotes('..\dowload_data\YDEX_1Day.txt')
# readQuotes('..\dowload_data\CNYRUBF_15Min_history.txt')
#readQuotes('..\dowload_data\YDEX_4HOUR.txt')
#readQuotes('..\dowload_data\SBER_1Day_history.txt')

df_history.tail()

,open,high,low,close,volume
Date,,,,,
2025-05-22,3990.0,4089.0,3925.0,4024.0,773429
2025-05-23,4032.0,4048.0,3996.0,4009.5,335512
2025-05-26,4002.5,4024.0,3886.0,3917.0,504972
2025-05-27,3917.5,4024.0,3855.0,3992.5,694856
2025-05-28,3998.0,4176.0,3995.0,4165.0,1192435


In [8]:
df_history['EMA'] = talib.EMA(df_history['close'], timeperiod=20)
df_history['SAR'] = parabolic_sar(df_history['high'], df_history['low'])

In [397]:
# ЛОНГ
# падающая -> растущая -> растущая
df_history['pattern'] = (df_history['close'].shift(2) < df_history['open'].shift(2)) & \
            (df_history['close'].shift(1) > df_history['open'].shift(1)) & \
            (df_history['close'] > df_history['open']) & \
            (
                (df_history['EMA'].shift(2) < df_history['high'].shift(2)) & \
                (df_history['EMA'].shift(1) < df_history['high'].shift(1)) & \
                (df_history['EMA'] < df_history['high'])
            )

max_ = 0
max_inx = 0
for i in range(2, N):
    df_history['next_2_candles'] = df_history['open'].shift(-1) < df_history['close'].shift(-i)

    st = df_history[df_history['pattern']]['next_2_candles'].mean()
    if max_ < st:
        max_ = st
        max_inx = i

if max_ > 0.49:
    print(max_inx, max_)

14 0.5714285714285714


In [398]:
# ШОРТ
# растущая -> падающая -> падающая
df_history['pattern'] = (df_history['close'].shift(2) > df_history['open'].shift(2)) & \
            (df_history['close'].shift(1) < df_history['open'].shift(1)) & \
            (df_history['close'] < df_history['open']) & \
            (
                (df_history['EMA'].shift(2) > df_history['high'].shift(2)) & \
                (df_history['EMA'].shift(1) > df_history['high'].shift(1)) & \
                (df_history['EMA'] >  df_history['high'])
            )

max_ = 0
max_inx = 0
for i in range(2, N):
    df_history['next_2_candles'] = df_history['open'].shift(-1) > df_history['close'].shift(-i)

    st = df_history[df_history['pattern']]['next_2_candles'].mean()
    if max_ < st:
        max_ = st
        max_inx = i

if max_ > 0.49:
    print(max_inx, max_)

13 0.6486486486486487


In [399]:
# ЛОНГ
# поглощение падющей свечи
df_history['pattern'] = (df_history['close'].shift(1) < df_history['open'].shift(1)) & \
            (df_history['open'] >= df_history['close'].shift(1)) & \
            (df_history['close'] >= df_history['open'].shift(1)) & \
            (
                (df_history['EMA'].shift(2) > df_history['low'].shift(2)) & \
                (df_history['EMA'].shift(1) > df_history['low'].shift(1)) & \
                (df_history['EMA'] > df_history['low'])
            )

max_ = 0
max_inx = 0
for i in range(2, N):
    df_history['next_2_candles'] = df_history['open'].shift(-1) < df_history['close'].shift(-i)
    # df_history['next_2_candles'] = (df_history['high'].shift(-i) - df_history['open'].shift(-1)) > 0.01

    st = df_history[df_history['pattern']]['next_2_candles'].mean()
    if max_ < st:
        max_ = st
        max_inx = i

if max_ > 0.29:
    print(max_inx, max_)

4 0.5294117647058824


In [400]:
# ШОРТ
# поглощение растущей свечи
df_history['pattern'] = (df_history['close'].shift(1) > df_history['open'].shift(1)) & \
            (df_history['open'] >= df_history['close'].shift(1)) & \
            (df_history['close'] <= df_history['open'].shift(1)) & \
            (
                (df_history['EMA'].shift(2) > df_history['high'].shift(2)) & \
                (df_history['EMA'].shift(1) > df_history['close'].shift(1)) & \
                (df_history['EMA'] >  df_history['high'])
            )

max_ = 0
max_inx = 0
for i in range(2, N):
    df_history['next_2_candles'] = df_history['open'].shift(-1) > df_history['close'].shift(-i)

    st = df_history[df_history['pattern']]['next_2_candles'].mean()
    if max_ < st:
        max_ = st
        max_inx = i

if max_ > 0.49:
    print(max_inx, max_)

2 0.5161290322580645


In [403]:
# ЛОНГ
# продлжение падения ( растет -> падает -> падает)
df_history['pattern'] = (df_history['close'].shift(2) > df_history['open'].shift(2)) & \
            (df_history['close'].shift(1) < df_history['open'].shift(1)) & \
            (df_history['close'] > df_history['open']) & \
            (
                (df_history['EMA'].shift(2) < df_history['low'].shift(2)) & \
                (df_history['EMA'].shift(1) < df_history['low'].shift(1)) & \
                (df_history['EMA'] <  df_history['low'])
            )

max_ = 0
max_inx = 0
for i in range(2, N):
    # df_history['next_2_candles'] = df_history['open'].shift(-1) < df_history['close'].shift(-i)
    df_history['next_2_candles'] = (
        df_history['high'].shift(-i) - df_history['open'].shift(-1)
        ) > 40

    st = df_history[df_history['pattern']]['next_2_candles'].mean()
    if max_ < st:
        max_ = st
        max_inx = i
        # df_history[df_history['pattern']]['next_2_candles']

if max_ > 0.19:
    print(max_inx, max_)

3 0.6842105263157895


In [102]:
df_history.tail(77)

,open,high,low,close,volume,EMA,SAR,pattern,next_2_candles
Date,,,,,,,,,
2025-03-01,4450.0,4455.0,4427.0,4438.0,13967,4421.068015,4709.621980,False,-154.0
2025-03-02,4430.5,4432.5,4365.0,4376.0,40558,4416.775823,4676.852222,False,-254.0
2025-03-03,4392.5,4399.5,4260.0,4347.5,1105947,4410.178126,4646.704044,False,-199.5
2025-03-04,4368.0,4619.5,4356.0,4586.0,1426446,4426.923066,4260.000000,False,139.5
2025-03-05,4585.5,4630.0,4470.0,4497.0,924568,4433.597060,4267.190000,True,141.0
...,...,...,...,...,...,...,...,...,...
2025-05-22,3990.0,4089.0,3925.0,4024.0,773429,4061.205966,3882.780625,False,NaN
2025-05-23,4032.0,4048.0,3996.0,4009.5,335512,4056.281588,3888.925012,False,NaN
2025-05-26,4002.5,4024.0,3886.0,3917.0,504972,4043.016675,4190.000000,False,NaN


In [410]:
df_history[df_history['pattern'] & df_history['next_2_candles']]['next_2_candles'].loc['2024-03-01':]

Date
2024-11-05    True
2024-11-18    True
2024-11-22    True
2025-04-10    True
Name: next_2_candles, dtype: bool

In [409]:
# ШОРТ
# продлжение падения (падает -> растет -> падает)
df_history['pattern'] = (df_history['close'].shift(2) < df_history['open'].shift(2)) & \
            (df_history['close'].shift(1) > df_history['open'].shift(1)) & \
            (df_history['close'] < df_history['open']) & \
            (
                (df_history['EMA'].shift(2) > df_history['high'].shift(2)) & \
                (df_history['EMA'].shift(1) > df_history['high'].shift(1)) & \
                (df_history['EMA'] >  df_history['high'])
            )

max_ = 0
max_inx = 0
for i in range(2, N):
    # df_history['next_2_candles'] = df_history['open'].shift(-1) > df_history['close'].shift(-i)
    df_history['next_2_candles'] = (
        df_history['open'].shift(-1) - df_history['low'].shift(-i)
    ) > 40

    st = df_history[df_history['pattern']]['next_2_candles'].mean()
    if max_ < st:
        max_ = st
        max_inx = i

if max_ >= 0.49:
    print(max_inx, max_)

2 0.5263157894736842


In [36]:
# ЛОНГ
# SAR под ценой, цена выше EMA
df_history['pattern'] = \
            (df_history['SAR'].shift(2) > df_history['high'].shift(2)) & \
            (df_history['SAR'].shift(1) < df_history['low'].shift(1)) & \
            (df_history['SAR'] < df_history['open']) & \
            (df_history['open'] > df_history['EMA'])

max_ = 0
max_inx = 0
for i in range(2, N):
    # df_history['next_2_candles'] = \
    #     df_history['open'].shift(-1) < df_history['close'].shift(-i)
    df_history['next_2_candles'] = \
        (df_history['close'].shift(-i) - df_history['open'].shift(-1)) #> 20

    st = df_history[df_history['pattern']]['next_2_candles'].mean()
    # print(st)
    if max_ < st:
        max_ = st
        max_inx = i
        # df_history[df_history['pattern']]['next_2_candles']

if max_ > 0.19:
    print(max_inx, max_)

12 8.719999999999981


In [37]:
df_history['next_2_candles'] = \
        (df_history['close'].shift(-6) - df_history['open'].shift(-1))
df_history[df_history['pattern']].tail(65)

,open,high,low,close,volume,EMA,SAR,pattern,next_2_candles
Date,,,,,,,,,
2023-11-29,2602.2,2628.8,2563.6,2574.0,902549,2589.643770,2542.388,True,-226.2
2024-06-06,4180.0,4238.0,4151.4,4204.2,261180,4153.476638,3896.512,True,-74.3
2025-01-31,4117.0,4154.0,4057.0,4092.0,758059,3986.152607,3927.970,True,136.0
2025-03-06,4498.0,4577.5,4480.0,4513.5,561254,4441.206864,4267.190,True,14.0
2025-05-14,4115.5,4154.0,4021.0,4021.0,364017,4088.789959,3821.520,True,20.0


In [92]:
df_history.tail(30)

,open,high,low,close,volume,EMA,SAR,pattern,next_2_candles
Date,,,,,,,,,
2025-04-23,4362.5,4382.0,4182.0,4285.0,1088953,4257.622729,3883.271898,False,265.0
2025-04-24,4287.0,4372.5,4245.0,4293.0,584541,4260.991993,3914.485584,False,206.5
2025-04-25,4301.5,4374.0,4294.0,4371.5,759468,4271.516565,3943.826449,False,216.5
2025-04-28,4322.5,4358.0,4237.0,4283.0,844808,4272.610226,3971.406862,False,231.0
2025-04-29,4285.0,4316.5,4131.5,4155.0,538880,4261.409252,3997.332450,False,113.5
2025-04-30,4154.0,4211.0,3993.0,4106.0,712587,4246.608371,4403.500000,False,122.5
2025-05-02,4113.0,4113.0,3900.0,3905.5,561471,4214.121859,4395.290000,False,-84.0
2025-05-03,3940.0,3960.0,3909.0,3940.5,61146,4188.062634,4375.478400,False,-64.5
2025-05-04,3945.0,3976.0,3940.0,3948.5,50717,4165.247145,4356.459264,False,38.0
